In [46]:
def fetch_image_data():
    # TODO
    return

In [47]:
from pathlib import Path
import os

IMAGES_PATH = Path("../resources/cloud-images/CCSN_v2")

def index_labeled_images(images_path=IMAGES_PATH):
    cloud_labels = []
    images_path = Path(images_path)
    labeled_images = {}
    if not images_path.exists():
        return labeled_images

    for cloud_dir in sorted(p for p in images_path.iterdir() if p.is_dir()):
        for img_path in sorted(p for p in cloud_dir.iterdir() if p.is_file()):
            labeled_images[img_path.name] = {
                "label": cloud_dir.name,
                "path": str(img_path)
            }

    return labeled_images, cloud_labels

In [48]:
labeled_images, cloud_labels = index_labeled_images()

In [49]:
import matplotlib.pyplot as plt
import numpy as np
import cv2

def extract_labels(labeled_images):
    images = []
    labels = []
    for (i, image_name) in enumerate(labeled_images):
        path = labeled_images[image_name]['path']
        image = cv2.imread(path)
        image = cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)
        image = cv2.resize(image, (64, 64))
        images.append(image)

        label = labeled_images[image_name]['label']
        labels.append(label)

    return np.array(images), np.array(labels)

In [50]:
images, labels = extract_labels(labeled_images)

In [ ]:
def split_train_test():
    # TODO with this and the below code blocks
    return

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder

ordinal_encoder = LabelEncoder()
labels_encoded = ordinal_encoder.fit_transform(labels)

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(sss.split(images, labels_encoded))

X_train, X_test = images[train_idx], images[test_idx]
y_train, y_test = labels_encoded[train_idx], labels_encoded[test_idx]

print("train:", X_train.shape, y_train.shape)
print("test: ", X_test.shape, y_test.shape)


train: (2034, 64, 64) (2034,)
test:  (509, 64, 64) (509,)


In [52]:
X_train_flat = X_train.reshape(len(X_train), -1)
X_test_flat  = X_test.reshape(len(X_test), -1)

In [53]:
def display_scores(scores):
    print("Scores:", scores)
    print("Mean:", scores.mean())
    print("Standard deviation:", scores.std())

In [54]:
from sklearn.metrics import precision_score, recall_score, f1_score

def display_clf_scores(labels, predictions):
    precision = precision_score(labels, predictions)
    recall = recall_score(labels, predictions)
    f1 = f1_score(labels, predictions)

    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)


In [ ]:
from typing import Dict, Any
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier


def get_default_classifiers(random_state: int = 42) -> Dict[str, Any]:
    """
    Return a dictionary of main classification models (some with scaling).
    """
    return {
        "gaussian_nb": GaussianNB(),
        "log_reg": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=random_state))
        ]),
        "linear_svc": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SGDClassifier(loss="hinge", random_state=random_state))
        ]),
        "svc_rbf": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(kernel="rbf", probability=True, random_state=random_state))
        ]),
        "knn": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier())
        ]),
        "decision_tree": DecisionTreeClassifier(random_state=random_state),
        "random_forest": RandomForestClassifier(random_state=random_state),
        "gradient_boosting": GradientBoostingClassifier(random_state=random_state,
                                                        n_estimators=150,
                                                        learning_rate=0.1,
                                                        max_depth=3,
                                                        subsample=0.7
                                                        ),
        "hist_gradient_boosting": HistGradientBoostingClassifier(random_state=random_state),
        "mlp": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(max_iter=500, random_state=random_state))
        ])
    }


def evaluate_classifiers(X, y, scoring: str = "accuracy", cv: int = 3,
                         n_jobs: int = -1, random_state: int = 42,
                         verbose: int = 1
) -> pd.DataFrame:
    """
    Evaluate many classification models with default hyperparameters.
    Returns a sorted dataframe of cross-validation results.
    """
    classifiers = get_default_classifiers(random_state)
    cv_obj = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)

    results = []

    for name, clf in classifiers.items():
        if verbose:
            print(f"\nEvaluating model: {name}")

        t0 = time.perf_counter()

        scores = cross_val_score(
            clf, X, y,
            scoring=scoring,
            cv=cv_obj,
            n_jobs=n_jobs
        )

        t1 = time.perf_counter()

        results.append({
            "model": name,
            "mean_score": np.mean(scores),
            "std_score": np.std(scores),
            "time_elapsed": t1 - t0
        })

    return pd.DataFrame(results).sort_values("mean_score", ascending=False)


In [75]:
results = evaluate_classifiers(X_train_flat, y_train, scoring="f1_macro", cv=5)
print(results)



Evaluating model: gaussian_nb

Evaluating model: log_reg

Evaluating model: linear_svc

Evaluating model: svc_rbf

Evaluating model: knn

Evaluating model: decision_tree

Evaluating model: random_forest

Evaluating model: gradient_boosting

Evaluating model: hist_gradient_boosting

Evaluating model: mlp
                    model  mean_score  std_score  time_elapsed
7       gradient_boosting    0.249540   0.010886    917.842738
8  hist_gradient_boosting    0.248760   0.005808    123.033588
6           random_forest    0.242161   0.015370      3.676388
9                     mlp    0.211470   0.026213      4.508299
3                 svc_rbf    0.196986   0.013427     12.453825
2              linear_svc    0.190364   0.011519      5.036427
4                     knn    0.187575   0.012554      1.006303
5           decision_tree    0.178849   0.012698      4.152716
0             gaussian_nb    0.165809   0.005751      1.258407
1                 log_reg    0.165537   0.011160      5.153570
